### Description

WIP. Scripts to convert the bioluminescence simulation files created by `fourth_day` into i3 files. It needs a working icetray environment. Now most of the paths are pointing to my (Cristina) directories in the `cobalt` machines in IceCube. 

In [ ]:
from icecube import dataio, icetray, dataclasses
from icecube.icetray import OMKey

import pickle as pkl
import numpy as np
import pandas as pd
import os
from optparse import OptionParser

usage = "usage: %prog [options]"
parser = OptionParser(usage)

parser.add_option(
    "-o",
    "--output_file",
    default="/home/clagunas/li/20modules_300s.i3",
    dest="OUTPUT_FILE",
    help="Path to output file",
)
parser.add_option(
    "--sim_path",
    default="/data/user/clagunas/li_data/20POMs/",
    dest="SIM_PATH",
    help="Path to the folder with simulation files",
)
parser.add_option(
    "-t",
    "--interpolation_time",
    type="float",
    default=0.1,
    dest="DELTA_TIME_S",
    help="Duration of each frame in s, default 0.1 s",
)

(options, args) = parser.parse_args()
print(options)

with open(os.path.join(options.SIM_PATH, "time_01.pkl"), "rb") as f:
    time = pkl.load(f)

# rename, combine and interpolate values
dfs = []

for i in range(1, 21):
    with open(os.path.join(options.SIM_PATH, f"detectors_{i:02d}.pkl"), "rb") as f:
        detectors = pkl.load(f)
    nameDict = {f"Detector {j}": f"{i:02d}_{j+1:02d}" for j in range(16)}
    detectors = detectors.rename(columns=nameDict)
    dfs.append(detectors)

final_df = pd.concat(dfs, axis=1)

print("Concatenated 20 modules")

# new index range for interpolation
times_new = np.arange(0, len(time), options.DELTA_TIME_S)

# interpolation for all columns at once
interpolated_data = {
    col: np.interp(times_new, time, final_df[col]) for col in final_df.columns
}

interpolated_df = pd.DataFrame(interpolated_data, index=times_new)

print(f"Interpolated time values, now {len(times_new)} values")

output_file = dataio.I3File(options.OUTPUT_FILE, "w")
offline_pmts = [
    7,
    8,
    5,
    6,
    3,
    4,
    1,
    2,
    14,
    13,
    16,
    15,
    11,
    10,
    9,
    12,
]  #  convert the biolum pmts to poneoffline pmts
string = 1  # only have 1 string now

for i, t in enumerate(times_new):

    pulse_series_map = dataclasses.I3RecoPulseSeriesMap()

    pmts = interpolated_df.iloc[i]
    non_zero_values = interpolated_df.iloc[i][interpolated_df.iloc[i] != 0.0].tolist()
    non_zero_detectors = [
        detector.split("_")
        for detector in interpolated_df.iloc[i][interpolated_df.iloc[i] != 0.0].index
    ]
    # column names as XX_YY where XX is module number and YY is PMT position

    # only add the omkeys for pmts that were hit to the pulse series
    if non_zero_detectors:
        for j, det in enumerate(non_zero_detectors):

            optical_module = int(det[0])
            pmt = offline_pmts[int(det[1]) - 1]

            # creates an empty series maps with only the omkeys of the pmt that were hit
            pulse_series_map[OMKey(string, optical_module, pmt)] = (
                dataclasses.I3RecoPulseSeries()
            )
            pulse = dataclasses.I3RecoPulse()
            pulse.time = t
            pulse.charge = non_zero_values[j]

            pulse_series_map[OMKey(string, optical_module, pmt)].append(pulse)

        # make an event header for this frame
        header = dataclasses.I3EventHeader()
        header.run_id = 0
        header.sub_run_id = 0
        header.event_id = i

        # make the frame and add everything
        pframe = icetray.I3Frame(icetray.I3Frame.DAQ)

        pframe["I3EventHeader"] = header
        pframe["Bioluminescence"] = pulse_series_map

        output_file.push(pframe)

output_file.close()
print(f"File saved in {options.OUTPUT_FILE}")